# Voice Preprocessing: Person + Language Classification

This notebook prepares the feature dataset for a model that predicts **which person is speaking** and **which language** they are speaking (Arabic, English, German, French).


## Imports
Libraries used for audio loading, feature extraction, and data handling.

In [ ]:
import librosa
import numpy as np
import os
import pandas as pd


## Extract Features

For every `.wav` file:
1. Load the audio at a fixed sample rate.
2. Trim leading/trailing silence.
3. Extract MFCC features (captures timbre/phonetic content — useful for both speaker identity and language, since different languages have distinct phoneme/rhythm patterns).
4. Normalize and average over time to get one fixed-length feature vector per file.
5. Record the `Person` label (from the parent folder) and `Language` label (from the sub-folder).

In [ ]:
data_path = "./Dataset"
SR = 22050          # sample rate to resample every file to
N_MFCC = 17          # number of MFCC coefficients
TOP_DB = 30          # silence threshold for trimming

Person = []
Language = []
Features = []

for person in os.listdir(data_path):
    person_path = os.path.join(data_path, person)
    if not os.path.isdir(person_path):
        continue

    for language in os.listdir(person_path):
        language_path = os.path.join(person_path, language)
        if not os.path.isdir(language_path):
            continue

        for voice in os.listdir(language_path):
            if not voice.endswith('.wav'):
                continue

            audio_path = os.path.join(language_path, voice)

            librosa_audio, sr = librosa.load(audio_path, sr=SR)              # load audio file
            removed_silence = librosa.effects.trim(librosa_audio, top_db=TOP_DB)[0]  # remove silence

            mfcc_features = librosa.feature.mfcc(y=removed_silence, sr=SR, n_mfcc=N_MFCC)  # extract MFCCs
            mfcc_norm = librosa.util.normalize(mfcc_features)                 # normalize
            mfcc_flattened = np.mean(mfcc_norm, axis=1)                       # collapse time axis

            Person.append(person)
            Language.append(language)
            Features.append(mfcc_flattened)

Features = np.array(Features)
Person = np.array(Person)
Language = np.array(Language)

print("Features shape:", Features.shape)
print("Person shape:", Person.shape)
print("Language shape:", Language.shape)
print("Languages found:", np.unique(Language))
print("Persons found:", np.unique(Person))


## Save the Preprocessed Dataset
Store the feature matrix and labels so the next notebook (model training) can load them directly, without re-running audio processing every time.

In [ ]:
df = pd.DataFrame(Features)
df.columns = [f"mfcc_{i+1}" for i in range(Features.shape[1])]
df["Person"] = Person
df["Language"] = Language

df.to_csv("preprocessed_features.csv", index=False)
print("Saved preprocessed_features.csv with shape:", df.shape)
